In [16]:
#Persiapan Data

In [18]:
import pandas as pd
import numpy as np

# Membaca dataset
df = pd.read_csv('paket_a_transaksi_pos_2024_2025.csv')

# Melihat data awal
print("Ukuran dataset:", df.shape)
display(df.head())
print("\\nTipe data setiap kolom:")
print(df.dtypes)

Ukuran dataset: (5030, 9)


,id_transaksi,tanggal,kode_gerai,kode_kasir,kode_produk,jumlah,harga_satuan,diskon_persen,status
0,TRX0241065,2025-06-02,G09,K011,RMT1402,4,13000,0,SELESAI
1,TRX0240510,2024-12-28,G09,K015,SGR1205,1,21000,0,SELESAI
2,TRX0244483,2024-09-04,G11,K029,RMT1407,3,"7500,00",0,SELESAI
3,TRX0242532,2025-07-30,G05,K001,MKN1103,2,6500,0,SELESAI
4,TRX0243509,26/03/2025,G05,K016,RMT1407,1,6000,0,SELESAI


\nTipe data setiap kolom:
id_transaksi       str
tanggal            str
kode_gerai         str
kode_kasir         str
kode_produk        str
jumlah           int64
harga_satuan       str
diskon_persen    int64
status             str
dtype: object


In [19]:
#Task 1 — Indexing & Selection

In [20]:
# Mengganti koma desimal menjadi titik dan mengubah kolom menjadi numerik
df['harga_satuan'] = pd.to_numeric(
    df['harga_satuan'].astype(str).str.replace(',', '.', regex=False),
    errors='coerce'
).astype(float)

print("Tipe data harga_satuan:", df['harga_satuan'].dtype)
display(df[['harga_satuan']].head())

Tipe data harga_satuan: float64


,harga_satuan
0,13000.0
1,21000.0
2,7500.0
3,6500.0
4,6000.0


In [21]:
df.loc[df['jumlah'] < 0, 'jumlah'] = df.loc[df['jumlah'] < 0, 'jumlah'].abs()

print("Jumlah nilai jumlah yang masih negatif:", (df['jumlah'] < 0).sum())
display(df[['id_transaksi', 'jumlah']].head())

Jumlah nilai jumlah yang masih negatif: 0


,id_transaksi,jumlah
0,TRX0241065,4
1,TRX0240510,1
2,TRX0244483,3
3,TRX0242532,2
4,TRX0243509,1


In [22]:
df.loc[df['kode_kasir'].isna(), 'kode_kasir'] = 'K_UNKNOWN'

print("Jumlah nilai kosong kode_kasir:", df['kode_kasir'].isna().sum())
print("Jumlah K_UNKNOWN:", (df['kode_kasir'] == 'K_UNKNOWN').sum())

Jumlah nilai kosong kode_kasir: 0
Jumlah K_UNKNOWN: 261


In [23]:
subset_iloc = df.iloc[:5, [0, 1, 2, 5]]
display(subset_iloc)

,id_transaksi,tanggal,kode_gerai,jumlah
0,TRX0241065,2025-06-02,G09,4
1,TRX0240510,2024-12-28,G09,1
2,TRX0244483,2024-09-04,G11,3
3,TRX0242532,2025-07-30,G05,2
4,TRX0243509,26/03/2025,G05,1


In [24]:
#Task 2 — Operasi Aritmatika (Feature Engineering)

In [25]:
df['Total_Kotor'] = df['harga_satuan'] * df['jumlah']
display(df[['id_transaksi', 'harga_satuan', 'jumlah', 'Total_Kotor']].head())

,id_transaksi,harga_satuan,jumlah,Total_Kotor
0,TRX0241065,13000.0,4,52000.0
1,TRX0240510,21000.0,1,21000.0
2,TRX0244483,7500.0,3,22500.0
3,TRX0242532,6500.0,2,13000.0
4,TRX0243509,6000.0,1,6000.0


In [26]:
df['Nominal_Diskon'] = df['Total_Kotor'] * (df['diskon_persen'] / 100)
display(df[['Total_Kotor', 'diskon_persen', 'Nominal_Diskon']].head())

,Total_Kotor,diskon_persen,Nominal_Diskon
0,52000.0,0,0.0
1,21000.0,0,0.0
2,22500.0,0,0.0
3,13000.0,0,0.0
4,6000.0,0,0.0


In [27]:
df['Total_Bersih'] = (df['Total_Kotor'] - df['Nominal_Diskon']) * 1.11
display(df[['Total_Kotor', 'Nominal_Diskon', 'Total_Bersih']].head())

,Total_Kotor,Nominal_Diskon,Total_Bersih
0,52000.0,0.0,57720.0
1,21000.0,0.0,23310.0
2,22500.0,0.0,24975.0
3,13000.0,0.0,14430.0
4,6000.0,0.0,6660.0


In [28]:
#Task 3 — Filtering (Penyaringan Data)

In [29]:
filter_multi = df[
    (df['status'] == 'SELESAI') &
    (df['kode_gerai'].isin(['G05', 'G09'])) &
    (df['Total_Bersih'] > 100000)
]

print("Jumlah transaksi yang memenuhi kriteria:", len(filter_multi))
display(filter_multi)

Jumlah transaksi yang memenuhi kriteria: 139


,id_transaksi,tanggal,kode_gerai,kode_kasir,kode_produk,jumlah,harga_satuan,diskon_persen,status,Total_Kotor,Nominal_Diskon,Total_Bersih
7,TRX0241251,2024-10-09,G09,K035,SGR1204,4,47500.0,10,SELESAI,190000.0,19000.0,189810.0
24,TRX0244253,2024-07-18,G09,K026,BYI1502,4,30500.0,0,SELESAI,122000.0,0.0,135420.0
25,TRX0244999,2024-08-09,G05,K015,MIN1003,4,28500.0,0,SELESAI,114000.0,0.0,126540.0
90,TRX0240898,2025-12-09,G05,K010,BYI1509,3,46500.0,0,SELESAI,139500.0,0.0,154845.0
206,TRX0241868,2025-07-01,G05,K009,SGR1203,6,20500.0,0,SELESAI,123000.0,0.0,136530.0
...,...,...,...,...,...,...,...,...,...,...,...,...
4877,TRX0243898,2024-04-23,G05,K008,SGR1204,12,51000.0,0,SELESAI,612000.0,0.0,679320.0
4911,TRX0241966,2024-04-22,G09,K024,SGR1206,3,42000.0,0,SELESAI,126000.0,0.0,139860.0
4925,TRX0240377,2025-06-11,G09,K028,XXX9969,12,51500.0,15,SELESAI,618000.0,92700.0,583083.0
4964,TRX0243219,2025-11-17,G09,K003,SGR1204,6,43500.0,0,SELESAI,261000.0,0.0,289710.0


In [30]:
filter_query = df.query('kode_gerai == "G10" and jumlah >= 5')
print("Jumlah baris:", len(filter_query))
display(filter_query)

Jumlah baris: 39


,id_transaksi,tanggal,kode_gerai,kode_kasir,kode_produk,jumlah,harga_satuan,diskon_persen,status,Total_Kotor,Nominal_Diskon,Total_Bersih
83,TRX0243880,2024-08-04,G10,K002,MKN1101,12,45500.0,0,SELESAI,546000.0,0.0,606060.0
166,TRX0241592,2024-05-11,G10,K033,SGR1202,6,13000.0,0,SELESAI,78000.0,0.0,86580.0
351,TRX0242686,2024-02-03,G10,K016,BYI1502,12,26000.0,0,SELESAI,312000.0,0.0,346320.0
556,TRX0241396,2025-12-16,G10,K009,RMT1406,12,5500.0,5,SELESAI,66000.0,3300.0,69597.0
879,TRX0243559,2025-09-30,G10,K016,SGR1206,6,47000.0,0,SELESAI,282000.0,0.0,313020.0
964,TRX0244939,2025-07-08,G10,K009,SGR1200,6,52000.0,10,SELESAI,312000.0,31200.0,311688.0
989,TRX0243076,2025-02-18,G10,K003,MKN1100,6,18000.0,0,SELESAI,108000.0,0.0,119880.0
1023,TRX0242437,2025-09-18,G10,K030,MKN1104,6,6000.0,0,SELESAI,36000.0,0.0,39960.0
1260,TRX0240537,2025-05-09,G10,K027,BYI1509,6,54000.0,0,SELESAI,324000.0,0.0,359640.0
1296,TRX0240698,2024-01-24,G10,K022,MKN1109,6,29000.0,0,SELESAI,174000.0,0.0,193140.0


In [32]:
#Task 4 — Sorting (Pengurutan Data)

In [33]:
sort_multi = df.sort_values(
    by=['kode_gerai', 'Total_Bersih'],
    ascending=[True, False]
)
display(sort_multi.head(10))

,id_transaksi,tanggal,kode_gerai,kode_kasir,kode_produk,jumlah,harga_satuan,diskon_persen,status,Total_Kotor,Nominal_Diskon,Total_Bersih
244,TRX0243257,2024-11-24,G01,K033,MKN1102,1456,32500.0,20,SELESAI,47320000.0,9464000.0,42020160.0
2956,TRX0241367,2025-05-15,G01,K006,RMT1408,12,50500.0,0,SELESAI,606000.0,0.0,672660.0
3588,TRX0240608,2025-07-25,G01,K006,MKN1101,12,47000.0,0,SELESAI,564000.0,0.0,626040.0
2937,TRX0240631,2024-12-24,G01,K013,RMT1408,12,42000.0,0,SELESAI,504000.0,0.0,559440.0
5,TRX0240658,2025-07-18,G01,K008,RMT1408,12,45500.0,10,SELESAI,546000.0,54600.0,545454.0
3889,TRX0241537,2024-10-19,G01,K013,MKN1102,12,27000.0,0,SELESAI,324000.0,0.0,359640.0
2179,TRX0244635,2025-08-30,G01,K009,BYI1503,6,52500.0,0,SELESAI,315000.0,0.0,349650.0
4270,TRX0240428,2024-03-31,G01,K004,RMT1408,6,48500.0,0,SELESAI,291000.0,0.0,323010.0
695,TRX0244706,2024-08-23,G01,K013,SGR1204,6,47000.0,0,BATAL,282000.0,0.0,313020.0
4404,TRX0242179,2024-02-11,G01,K019,RMT1408,6,47000.0,0,SELESAI,282000.0,0.0,313020.0


In [34]:
top_5 = df.nlargest(5, 'Total_Bersih')
display(top_5[['id_transaksi', 'kode_gerai', 'jumlah', 'Total_Bersih', 'status']])

,id_transaksi,kode_gerai,jumlah,Total_Bersih,status
994,TRX0241002,G11,2093,100363536.0,SELESAI
4985,TRX0244221,G09,1301,66934498.5,SELESAI
467,TRX0243696,G06,892,53466480.0,SELESAI
3393,TRX0241029,G07,1983,45563391.0,SELESAI
244,TRX0243257,G01,1456,42020160.0,SELESAI


In [35]:
df_selesai = df[df['status'] == 'SELESAI']

laporan_gerai = df_selesai.groupby('kode_gerai').agg(
    Jumlah_Transaksi=('id_transaksi', 'count'),
    Total_Unit_Terjual=('jumlah', 'sum'),
    Total_Omzet_Bersih=('Total_Bersih', 'sum'),
    Rata_Rata_Transaksi=('Total_Bersih', 'mean')
).sort_values(
    by='Total_Omzet_Bersih',
    ascending=False
).round(2)

display(laporan_gerai)

,Jumlah_Transaksi,Total_Unit_Terjual,Total_Omzet_Bersih,Rata_Rata_Transaksi
kode_gerai,,,,
G11,398,3170,1.253536e+08,314958.82
G06,383,3103,9.113936e+07,237961.77
G09,379,3027,9.019606e+07,237984.32
G07,384,4284,8.036181e+07,209275.55
G01,450,2635,7.077959e+07,157287.97
G03,372,3274,5.980970e+07,160778.77
G10,401,1742,5.892248e+07,146938.85
G08,408,1700,3.550256e+07,87016.08
G12,406,1010,2.464483e+07,60701.55
